## 检查商品明细表 

In [ ]:
import pandas as pd 
from pathlib import Path 

project_root = Path.cwd().parent 
data_path = project_root.joinpath('data','raw') 
print(data_path)

In [ ]:
items = pd.read_csv(data_path.joinpath('olist_order_items_dataset.csv')) 

In [ ]:
# 检查商品明细 

print(items.shape) 

# print(items.head()) 

# 查看字符类型
# items.info() 

# print(items.isna().sum()) 

print('价格为负数的数量:', (items['price']<0).sum()) 

print('价格为零的数量:', (items['price']==0).sum())  

print('运费为负数的数量:', (items['freight_value']<0).sum()) 

print('运费为零的数量:', (items['freight_value']==0).sum()) 

# 检查组合键是否重复 
print('订单明细组合键重复数量:', (items.duplicated(subset=['order_id', 'order_item_id'])).sum())

print('各字段缺失值:')
print(items.isna().sum()) 

In [ ]:
# 订单商品明细聚合 

order_items_agg = (
    items.groupby('order_id', as_index=False) 
    #聚合完成后，让 order_id 保留为普通列，而不是成为行索引。
    .agg(
        item_count=('order_item_id', 'size'), 
        #这里的 size 统计的是行数，包括指定列中有缺失值的行；而 count 只统计非缺失值
        product_amount=('price', 'sum'), 
        freight_amount=('freight_value', 'sum') 
    )
)

# 计算订单总金额(商品金额+运费) 
order_items_agg['total_amount'] = (
    order_items_agg['product_amount'] 
    + order_items_agg['freight_amount']
)

print('聚合后的数据规模:', order_items_agg.shape)

print(order_items_agg.head()) 


In [ ]:
print('原始明细行数:', len(items)) 

print('不同订单数量:', items['order_id'].nunique()) 

print('聚合后行数:', len(order_items_agg))   



In [ ]:
# 金额一致性检查

import numpy as np 


# 检查聚合后的订单编号是否唯一 
print(
    '聚合后重复的订单数:', 
    order_items_agg['order_id'].duplicated().sum() 
) 

# 检查商品金额 
original_price = items['price'].sum() 
aggregated_price = order_items_agg['product_amount'].sum() 

print('原始商品金额:', original_price) 
print('聚合后的商品金额:', aggregated_price) 
print('商品金额一致性:', 
      np.isclose(original_price, aggregated_price)) #np.isclose() 判断两个数在一定容差范围内是否接近

# 检查运费 
original_freight = items['freight_value'].sum() 
aggregated_freight = order_items_agg['freight_amount'].sum() 

print('原始运费:', original_freight) 
print('聚合后的运费:', aggregated_freight) 
print(
    '运费一致性:', 
    np.isclose(original_freight, aggregated_freight)
)

# 商品明细数是否一致
original_item = len(items)
aggregated_item = order_items_agg['item_count'].sum()  
print('原始商品明细数量:', original_item) 
print('聚合后商品明细数量:', aggregated_item)
print(
    '商品明细数量一致性:', 
    np.isclose(original_item, aggregated_item)
)


In [ ]:
# 将聚合结果保存 
processed_path = project_root.joinpath('data', 'processed') 

# 如果目录不存在,就创建 
processed_path.mkdir(parents=True, exist_ok=True) 
# parents=True：允许创建多级目录
# exist_ok=True：文件夹已存在时不报错


# 设置输出文件路径 
output_path = processed_path.joinpath('order_items_agg.csv') 
# 将聚合后的DataFrame保存 
order_items_agg.to_csv(output_path, index=False) 

# 检查文件是否保存成功 
print('保存路径:', output_path) 
print('文件是否存在:',output_path.is_file()) 



In [ ]:
# 重新读取并检查
saved_items = pd.read_csv(output_path) 

print('保存后的数据规模:', saved_items.shape) 

print(
    '数据规模是否一致:', 
    saved_items.shape == order_items_agg.shape 
)

print(
    '列名是否一致:', 
    saved_items.columns.equals(order_items_agg.columns)
)

print(
    '订单编号是否唯一:', 
    saved_items['order_id'].is_unique 
)

display(saved_items.head()) 

In [ ]:
report_path = project_root.joinpath('reports') 
report_path.mkdir(parents=True, exist_ok=True) 
